# **Proposed Thesis**
**Natural Language Processing (NLP)-Based Analysis of Gulf Cooperation Council (GCC) Gen Z Consumer Responses to AI-Generated versus Human-Created Digital Marketing Content**

**Data Preprocessing(EDA)**

In [7]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
nltk.download("vader_lexicon")

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [9]:
# RENAME COLUMNS
colRenames_ = {
    "#": "resp_id",
    "I have read the above and consent to participate.": "consent",
    "Q1. What is your current age?": "age_group",
    "Q2. What is your country of residence?": "country",
    "Q3. Gender": "gender",
    "Q4. Estimated total monthly spend on online purchases (BHD)": "monthly_spend",
    "Q5. How often do you purchase through e-commerce sites or apps (e.g., Amazon, Noon, Shein) *every six months*?": "ecom_freq",
    "Q6. How often do you purchase directly through social media (e.g., Instagram Shop, TikTok Shop) *every six months*?": "social_freq",
    "Q7. I am familiar with AI-generated content (e.g., ads, images, captions created by AI tools)?": "ai_familiarity",
    "Q8. I can tell the difference between an AI-generated Advertisement and a human-created Advertisement?": "ai_detection",
    "Q9. I trust AI-generated advertisements as much as I trust human-created advertisement.": "ai_trust",
    "Q10. Knowing an Advertisement was made by AI would make me less likely to buy the product.": "ai_buy_deterrent",
    "Q11. Personalised advertisements make me feel my privacy is being invaded.": "privacy_invasion",
    "I feel uneasy when I discover an Advertisement tailored to my personal information.": "privacy_unease",
    "Q13. Which Burger advertisement feels more authentic and trustworthy?": "burger_auth",
    "Q14. Which Burger advertisement makes you more likely to purchase the product?": "burger_purchase",
    "Q15. Which Burger advertisement do you think was created by AI?": "burger_ai_guess",
    "I preferred *Advertisement A* because it felt more realistic and relatable": "burger_pref_a_realistic",
    "I preferred *Advertisement A* because it felt more personal and trustworthy": "burger_pref_a_personal",
    "I preferred *Advertisement B* because it looked more professionally produced": "burger_pref_b_professional",
    "I preferred *Advertisement B* because it was more persuasive and compelling": "burger_pref_b_persuasive",
    "Q17. Which Barbie phone advertisement feels more authentic and trustworthy?": "barbie_auth",
    "Q18. Which Barbie phone advertisement makes you more likely to purchase the product?": "barbie_purchase",
    "Q19. Which Barbie phone advertisement do you think was created by AI?": "barbie_ai_guess",
    "I preferred *Advertisement A* because it felt more realistic and relatable.1": "barbie_pref_a_realistic",
    "I preferred *Advertisement A* because it felt more personal and trustworthy.1": "barbie_pref_a_personal",
    "I preferred *Advertisement B* because it looked more professionally produced.1": "barbie_pref_b_professional",
    "I preferred *Advertisement B* because it was more persuasive and compelling.1": "barbie_pref_b_persuasive",
    "Advertisement Set 3 — Product: clothing": "clothing_ad_set",
    "Q21. Which clothing advertisement feels more authentic and trustworthy?": "clothing_auth",
    "Q22. Which clothing advertisement makes you more likely to purchase the product?": "clothing_purchase",
    "Q23. Which clothing advertisement do you think was created by AI?": "clothing_ai_guess",
    "I preferred *Advertisement A* because it felt more realistic and relatable.2": "clothing_pref_a_realistic",
    "I preferred *Advertisement A* because it felt more personal and trustworthy.2": "clothing_pref_a_personal",
    "I preferred *Advertisement B* because it looked more professionally produced.2": "clothing_pref_b_professional",
    "I preferred *Advertisement B* because it was more persuasive and compelling.2": "clothing_pref_b_persuasive",
    "Q25. If an advertisement is clearly labelled as 'Created with AI,' my purchase intention would:": "ai_label_effect",
    "Emotional AI advertisements impact my purchases more than neutral ones.": "emotion_impact",
    "Q27. I am more likely to buy a product when the advertisement was shown at a moment I was already browsing related content.": "timing_effect",
    "Q28. When a product in an advertisement looks better than it would in real life due to AI-edited visuals, my intention to buy decreases.": "visual_misrep_effect",
    "Q29. Overall, do digital advertisements influence your desire to purchase products?": "digital_ad_influence",
    "Q30. Overall, which type of advertisement content do you find more persuasive?": "preferred_content_type",
    "Q31. Is there anything else you would like to share about how AI-powered marketing affects your decisions?": "open_comment",
    "Response Type": "response_status",
    "Start Date (UTC)": "start_utc",
    "Stage Date (UTC)": "stage_utc",
    "Submit Date (UTC)": "submit_utc",
    "Network ID": "network_id",
    "Tags": "tags",
    "Ending": "ending_msg"
}

#LOADING DATA
print(" LOADING DATA")
datasetPath = "/content/survey_raw.xlsx"
dataDF = pd.read_excel(datasetPath)
print(f"Original dataset shape: {dataDF.shape}")
# Column renaming
dataDF.rename(columns=colRenames_, inplace=True)
# REMOVE INCOMPLETE RESPONSES
print(" REMOVING INCOMPLETE RESPONSES")
dataDF = dataDF.dropna(thresh=int(dataDF.shape[1] * 0.5))
print(f"After removing incomplete responses: {dataDF.shape}")
# REMOVE ADMIN COLUMNS
print(" REMOVING ADMIN COLUMNS")
removeCols_ = [
    "response_status",
    "start_utc",
    "stage_utc",
    "submit_utc",
    "network_id",
    "tags",
    "ending_msg"
]

dataDF = dataDF.drop(columns=[c for c in removeCols_ if c in dataDF.columns], errors="ignore")
print(f"After removing admin columns: {dataDF.shape}")

# TEXT CLEANING FUNCTION
def textCleaningFun(txt):
    if pd.isna(txt):
        return ""
    txt = str(txt).lower()
    txt = re.sub(r"http\S+", "", txt)
    txt = re.sub(r"[^a-zA-Z\s]", "", txt)
    txt = re.sub(r"\s+", " ", txt)
    return txt.strip()

# SELECTING TEXT COLUMNS
print(" SELECTING TEXT COLUMNS")
preference_columns = []
textCols_ = []
setKeywords = ["preferred", "advertisement", "authentic", "trust", "persuasive",
               "realistic", "personal", "professional", "share"]

for col in dataDF.columns:
    if any(k.lower() in col.lower() for k in setKeywords):
        preference_columns.append(col)

textCols_ = preference_columns + ["open_comment"]
print(f"Text columns used ({len(textCols_)}):")
for i in textCols_:
    print(f"  - {i}")

# GENERATE NLP TEXT DATA
print(" CREATING SENTIMENT DATASET")
sentimentDF_ = pd.DataFrame()
sentimentDF_["text"] = dataDF[textCols_].fillna("").astype(str).apply(lambda x: " ".join(x), axis=1)
sentimentDF_["text"] = sentimentDF_["text"].apply(textCleaningFun)
# Remove empty text rows
sentimentDF_ = sentimentDF_[sentimentDF_["text"] != ""]
print(f"Sentiment dataset shape: {sentimentDF_.shape}")

# SENTIMENT LABEL CREATION (KEYWORD METHOD)
print(" SENTIMENT LABEL CREATION (KEYWORD METHOD)")
setPositiveWords = [
    "authentic", "trustworthy", "realistic", "relatable", "personal",
    "professional", "persuasive", "creative", "engaging", "better",
    "good", "great", "excellent", "amazing", "wonderful", "impressive",
    "love", "like", "enjoy", "appreciate", "recommend", "satisfied"
]

setNegativeWords = [
    "hate", "fake", "less likely", "invasive", "privacy",
    "uneasy", "decrease", "bad", "dislike", "untrustworthy",
    "unreliable", "deceptive", "misleading", "artificial",
    "unappealing", "unconvincing", "poorly", "unprofessional",
    "unrealistic", "manipulative", "suspicious", "creep",
    "ick", "lifeless", "inauthentic", "dishonest", "fraud",
    "scam", "cheap", "low effort", "not real", "doesn't feel real",
    "question", "doubt", "worry", "concern", "problem"
]

setNeutralWords = [
    "both", "equally", "same", "difference", "cannot tell",
    "no difference", "depends", "maybe", "sometimes", "neutral"
]

def generateSentiments(text):
    getPos = sum(w in text for w in setPositiveWords)
    getNeg = sum(w in text for w in setNegativeWords)
    getNeutral = sum(w in text for w in setNeutralWords)

    if getPos > getNeg and getPos > getNeutral:
        return "positive"
    elif getNeg > getPos and getNeg > getNeutral:
        return "negative"
    else:
        return "neutral"

sentimentDF_["sentiment_keyword"] = sentimentDF_["text"].apply(generateSentiments)
sentimentDF_["label_keyword"] = sentimentDF_["sentiment_keyword"].map({"negative": 0, "neutral": 1, "positive": 2})

print("Keyword-based Sentiment distribution:")
print(sentimentDF_["sentiment_keyword"].value_counts())

# VADER SENTIMENT ANALYSIS (MORE ACCURATE)
print(" VADER SENTIMENT ANALYSIS")
# Initialize VADER
sia = SentimentIntensityAnalyzer()
def getVaderSentiment(text):
    scores = sia.polarity_scores(text)
    comp = scores['compound']

    if comp >= 0.05:
        return "positive"
    elif comp <= -0.05:
        return "negative"
    else:
        return "neutral"

sentimentDF_["sentiment_vader"] = sentimentDF_["text"].apply(getVaderSentiment)
sentimentDF_["label_vader"] = sentimentDF_["sentiment_vader"].map({"negative": 0, "neutral": 1, "positive": 2})

print("VADER Sentiment distribution:")
print(sentimentDF_["sentiment_vader"].value_counts())

# Compare the two methods
print("\nComparison of Methods (Keyword vs VADER):")
comparison = pd.crosstab(sentimentDF_['sentiment_keyword'], sentimentDF_['sentiment_vader'])
print(comparison)

# CHOOSE BEST METHOD (VADER is more accurate)
print(" USING VADER SENTIMENT LABELS")
# Use VADER labels as the primary sentiment
sentimentDF_["sentiment"] = sentimentDF_["sentiment_vader"]
sentimentDF_["label"] = sentimentDF_["label_vader"]
print("Final Sentiment distribution (VADER):")
print(sentimentDF_["sentiment"].value_counts())

# HANDCRAFTED FEATURES
print(" CREATING HANDCRAFTED FEATURES")
authenticityWords = ["authentic", "trust", "realistic", "relatable", "genuine"]
emotionWords = ["love", "hate", "good", "bad", "trust", "uneasy", "creative"]
personalisationWords = ["personal", "tailored", "custom", "recommend"]
privacyWords = ["privacy", "invaded", "information", "data"]
def countWordsFun(text, ws):
    return sum(text.count(w) for w in ws)

featureDF_ = sentimentDF_.copy()
featureDF_["authenticity_score"] = featureDF_["text"].apply(lambda x: countWordsFun(x, authenticityWords))
featureDF_["emotion_score"] = featureDF_["text"].apply(lambda x: countWordsFun(x, emotionWords))
featureDF_["personalisation_score"] = featureDF_["text"].apply(lambda x: countWordsFun(x, personalisationWords))
featureDF_["privacy_score"] = featureDF_["text"].apply(lambda x: countWordsFun(x, privacyWords))

print("Handcrafted features added successfully!")

# ADD LIKERT VARIABLES
print(": ADDING LIKERT VARIABLES")
likertCols = ["ai_familiarity", "ai_detection", "ai_trust", "ai_buy_deterrent"]
for col in likertCols:
    if col in dataDF.columns:
        featureDF_[col] = dataDF.loc[featureDF_.index, col]

print(f"Added {len(likertCols)} Likert variables")
for c in likertCols:
    print(f"  - {c}")

# SAVE ALL FILES
print(": SAVING DATASETS")
# Save cleaned survey
dataDF.to_excel("survey_cleaned.xlsx", index=False)
print(" survey_cleaned.xlsx saved")
# Save sentiment dataset
sentimentDF_.to_excel("sentiment_dataset.xlsx", index=False)
print("sentiment_dataset.xlsx saved")
# Save feature dataset
featureDF_.to_excel("feature_sentiment_dataset.xlsx", index=False)
print(" feature_sentiment_dataset.xlsx saved")
#SUMMARY
print("FINAL SUMMARY")
print("\n Dataset Statistics:")
print(f"  Total responses: {len(dataDF)}")
print(f"  Text columns used: {len(textCols_)}")
print(f"  Sentiment dataset: {len(sentimentDF_)} samples")
print(f"  Feature dataset: {len(featureDF_)} samples, {len(featureDF_.columns)} features")
print("\n Sentiment Distribution (VADER - Final):")
sentiCounts = sentimentDF_['sentiment'].value_counts()
for s, c in sentiCounts.items():
    p = (c / len(sentimentDF_)) * 100
    print(f"  {s.capitalize()}: {c} ({p:.1f}%)")

print("\n Sentiment Distribution (Keyword Method - For Comparison):")
keyword_counts = sentimentDF_['sentiment_keyword'].value_counts()
for s, c in keyword_counts.items():
    p = (c / len(sentimentDF_)) * 100
    print(f"  {s.capitalize()}: {c} ({p:.1f}%)")

print("\n Features Created:")
featureNames = featureDF_.columns.tolist()
for id, f in enumerate(featureNames, 1):
    print(f"  {id}. {f}")



 LOADING DATA


/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Original dataset shape: (496, 50)
 REMOVING INCOMPLETE RESPONSES
After removing incomplete responses: (276, 50)
 REMOVING ADMIN COLUMNS
After removing admin columns: (276, 43)
 SELECTING TEXT COLUMNS
Text columns used (15):
  - ai_trust
  - burger_pref_a_realistic
  - burger_pref_a_personal
  - burger_pref_b_professional
  - burger_pref_b_persuasive
  - barbie_pref_a_realistic
  - barbie_pref_a_personal
  - barbie_pref_b_professional
  - barbie_pref_b_persuasive
  - clothing_pref_a_realistic
  - clothing_pref_a_personal
  - clothing_pref_b_professional
  - clothing_pref_b_persuasive
  - preferred_content_type
  - open_comment
 CREATING SENTIMENT DATASET
Sentiment dataset shape: (276, 1)
 SENTIMENT LABEL CREATION (KEYWORD METHOD)
Keyword-based Sentiment distribution:
sentiment_keyword
positive    267
neutral       9
Name: count, dtype: int64
 VADER SENTIMENT ANALYSIS
VADER Sentiment distribution:
sentiment_vader
positive    255
negative     12
neutral       9
Name: count, dtype: int64

